In [ ]:
import pandas as pd
from collections import Counter

# Define land cover mapping
land_cover_mapping = {
    '1': 'Developed',
    '2': 'Cropland',
    '3': 'Grass/Shrub',
    '0': 'Tree Cover',  # Recoded from '4'
    '5': 'Water',
    '6': 'Wetland',
    '7': 'Ice/Snow',
    '8': 'Barren'
}

# Load the CSV file
file_path = 'G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Edge_adjunct_LC\\Ecoregion_Classification\\eco_1985.csv'
data = pd.read_csv(file_path)

data = data.drop(columns=['NAME'])

def decode_landcover(code):
    """ Decode the five-digit landcover code into readable labels for each side. """
    code_str = str(int(code)).zfill(5)
    return [
        land_cover_mapping.get(digit, 'Unknown') for digit in code_str[1:] if digit != '0'
    ]

# Prepare a dictionary to store the results
ecoregion_landcover_counts = {}

# Group data by ecoregion
grouped_data = data.groupby('NA_L1NAME').sum()

In [ ]:
grouped_data

In [ ]:
# Iterate over each ecoregion group
for name, group in grouped_data.iterrows():
    # Initialize a Counter for this ecoregion
    ecoregion_counter = Counter()
    
    # Process each column in the group (excluding 'NA_L1NAME' which is the index now)
    for col in group.index[1:]:  # Adjust if the index is not set correctly
        # Decode the column header to get land cover types
        land_covers = decode_landcover(col)
        
        # Get the sum of pixels for this column in the current ecoregion
        pixel_sum = group[col]
        
        # Count each land cover type, weighted by the pixel sum
        land_cover_count = Counter(land_covers)  # Count how many times each land cover appears
        for land_cover, count in land_cover_count.items():
            if land_cover != 'Tree Cover':  # Exclude 'Tree Cover'
                ecoregion_counter[land_cover] += pixel_sum * count  # Multiply by both pixel sum and occurrences
    
    # Store the result for this ecoregion
    ecoregion_landcover_counts[name] = ecoregion_counter

# Output the results
for ecoregion, counts in ecoregion_landcover_counts.items():
    print(f"Ecoregion: {ecoregion}, Counts: {counts}")


In [ ]:
decode_landcover(13266)

In [ ]:
Counter(decode_landcover(13266))

In [ ]:
import pandas as pd
import os
from tqdm import trange
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt

# Define land cover mapping
land_cover_mapping = {
    '1': 'Developed',
    '2': 'Cropland',
    '3': 'Grass/Shrub',
    '0': 'Tree Cover',  # Recoded from '4'
    '5': 'Water',
    '6': 'Wetland',
    '7': 'Ice/Snow',
    '8': 'Barren'
}

def decode_landcover(code):
    """ Decode the five-digit landcover code into readable labels for each side. """
    code_str = str(int(code)).zfill(5)
    return [
        land_cover_mapping.get(digit, 'Unknown') for digit in code_str[1:] if digit != '0'
    ]

# Directory path
dir_path = "G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Edge_adjunct_LC\\Ecoregion_Classification"

# Process all files and store results
results = {}
for year in trange(1985, 2022):
    file_path = os.path.join(dir_path, f"eco_{year}.csv")
    if os.path.exists(file_path):
        data = pd.read_csv(file_path)
        data = data.drop(columns=[col for col in ['NAME', 'shape_id'] if col in data.columns])
        
        # Group data by ecoregion and sum
        grouped_data = data.groupby('NA_L1NAME').sum()
        
        # Count land covers by ecoregion
        ecoregion_landcover_counts = {}
        for name, group in grouped_data.iterrows():
            ecoregion_counter = Counter()
            for col in group.index:
                land_covers = decode_landcover(col)
                pixel_sum = group[col]
                land_cover_count = Counter(land_covers)
                for land_cover, count in land_cover_count.items():
                    if land_cover != 'Tree Cover':
                        ecoregion_counter[land_cover] += pixel_sum * count
            ecoregion_landcover_counts[name] = ecoregion_counter
        results[year] = ecoregion_landcover_counts
    else:
        print(f"File not found for the year {year}")

In [ ]:
# Visualize total land cover types by year
total_counts_by_year = {}
for year, data in results.items():
    year_counter = Counter()
    for ecoregion_counts in data.values():
        year_counter.update(ecoregion_counts)
    total_counts_by_year[year] = year_counter

# Plot results
plt.figure(figsize=(10, 5))
land_covers = ['Developed', 'Cropland', 'Grass/Shrub', 'Water', 'Wetland', 'Ice/Snow', 'Barren']
styles = ['-', '--', '-.', ':', '-', '--', '-.']  # Ensure this list matches the number of land covers

for land_cover, style in zip(land_covers, styles):
    plt.plot(list(total_counts_by_year.keys()), [counts.get(land_cover, 0) for counts in total_counts_by_year.values()], style, label=land_cover, lw =2)
plt.xlabel('Year')
plt.ylabel('Count')
plt.title('Total Adjunct Land Cover by Year')
plt.legend()
plt.show()


In [ ]:
# Visualize total land cover types by year
total_counts_by_year = {}
for year, data in results.items():
    year_counter = Counter()
    for ecoregion_counts in data.values():
        year_counter.update(ecoregion_counts)
    total_counts_by_year[year] = year_counter

# Calculate total counts for all land covers combined, per year
total_all_land_covers_by_year = {year: sum(counts.values()) for year, counts in total_counts_by_year.items()}

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(
    list(total_all_land_covers_by_year.keys()), 
    list(total_all_land_covers_by_year.values()), 
    label='Total Adjunct Land Cover Pixel Number', 
    lw=2, 
    marker='o',  # Optional: adds markers to the line
    linestyle='-'  # Defines the line style
)
plt.xlabel('Year')
plt.ylabel('Total Count')
plt.title('Total Adjunct Land Cover Pixel Number by Year')
plt.legend()
plt.grid(True)  # Optional: adds a grid for better readability
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extract a unique list of all ecoregions from the results
ecoregions = set()
for data in results.values():
    ecoregions.update(data.keys())

# Sort the list for consistent ordering
ecoregions = sorted(ecoregions)

# Define land covers to plot
land_covers = ['Developed', 'Cropland', 'Grass/Shrub', 'Water', 'Wetland', 'Ice/Snow', 'Barren']
styles = ['-', '--', '-.', ':', '-', '--', '-.']  # Ensure this list matches the number of land covers

# Calculate rows needed for the number of ecoregions
num_rows = len(ecoregions) // 4 + (1 if len(ecoregions) % 4 > 0 else 0)

# Setup subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=4, figsize=(20, 3 * num_rows), sharex=True)
axes = axes.flatten()  # Flatten the axes array to simplify indexing

# Loop through each ecoregion and each subplot axis
for ax, ecoregion in zip(axes, ecoregions):
    # Plot each land cover type for this ecoregion
    for land_cover, style in zip(land_covers, styles):
        ax.plot(
            list(total_counts_by_year.keys()), 
            [data.get(ecoregion, {}).get(land_cover, 0) for data in results.values()],
            style, 
            label=land_cover,
            lw = 2
        )
    ax.set_title(f'Land Cover Trends in {ecoregion}')
    ax.set_ylabel('Count')
    ax.legend()

# Disable unused axes if ecoregions do not fill up the entire grid
for i in range(len(ecoregions), len(axes)):
    axes[i].axis('off')

# Set common labels
axes[-1].set_xlabel('Year')  # Set x-label on the last subplot (common x-axis)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extract a unique list of all ecoregions from the results
ecoregions = set()
for data in results.values():
    ecoregions.update(data.keys())

# Sort the list for consistent ordering
ecoregions = sorted(ecoregions)

# Define the land cover to plot
land_cover_to_plot = 'Cropland'

# Calculate rows needed for the number of ecoregions
num_rows = len(ecoregions) // 4 + (1 if len(ecoregions) % 4 > 0 else 0)

# Setup subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=4, figsize=(20, 3 * num_rows), sharex=True)
axes = axes.flatten()  # Flatten the axes array to simplify indexing

# Loop through each ecoregion and each subplot axis
for ax, ecoregion in zip(axes, ecoregions):
    # Plot only 'Cropland' land cover type for this ecoregion
    ax.plot(
        list(total_counts_by_year.keys()), 
        [data.get(ecoregion, {}).get(land_cover_to_plot, 0) for data in results.values()],
        label=land_cover_to_plot,
        lw=2,  # Line width
        linestyle='-',  # Line style
        marker='o'  # Marker type
    )
    ax.set_title(f'Cropland Trends in {ecoregion}')
    ax.set_ylabel('Count')
    ax.legend()

# Disable unused axes if ecoregions do not fill up the entire grid
for i in range(len(ecoregions), len(axes)):
    axes[i].axis('off')

# Set common labels
axes[-1].set_xlabel('Year')  # Set x-label on the last subplot (common x-axis)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract a unique list of all ecoregions from the results
ecoregions = set()
for data in results.values():
    ecoregions.update(data.keys())

# Sort the list for consistent ordering
ecoregions = sorted(ecoregions)

# Define the land cover to plot
land_cover_to_plot = 'Grass/Shrub'

# Calculate rows needed for the number of ecoregions
num_rows = len(ecoregions) // 4 + (1 if len(ecoregions) % 4 > 0 else 0)

# Setup subplots
fig, axes = plt.subplots(dpi = 200, nrows=num_rows, ncols=4, figsize=(20, 3 * num_rows), sharex=True)
axes = axes.flatten()  # Flatten the axes array to simplify indexing

# Loop through each ecoregion and each subplot axis
for ax, ecoregion in zip(axes, ecoregions):
    ax.plot(
        list(total_counts_by_year.keys()), 
        [data.get(ecoregion, {}).get(land_cover_to_plot, 0) for data in results.values()],
        label=land_cover_to_plot,
        lw=2,  # Line width
        linestyle='-',  # Line style
        marker='o'  # Marker type
    )
    ax.set_title(f'Grass/Shrub Trends in {ecoregion}')
    ax.set_ylabel('Count')
    ax.legend()

# Disable unused axes if ecoregions do not fill up the entire grid
for i in range(len(ecoregions), len(axes)):
    axes[i].axis('off')

# Set common labels
axes[-1].set_xlabel('Year')  # Set x-label on the last subplot (common x-axis)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract a unique list of all ecoregions from the results
ecoregions = set()
for data in results.values():
    ecoregions.update(data.keys())

# Sort the list for consistent ordering
ecoregions = sorted(ecoregions)

# Define the land cover to plot
land_cover_to_plot = 'Developed'

# Calculate rows needed for the number of ecoregions
num_rows = len(ecoregions) // 4 + (1 if len(ecoregions) % 4 > 0 else 0)

# Setup subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=4, figsize=(20, 3 * num_rows), sharex=True)
axes = axes.flatten()  # Flatten the axes array to simplify indexing

# Loop through each ecoregion and each subplot axis
for ax, ecoregion in zip(axes, ecoregions):
    ax.plot(
        list(total_counts_by_year.keys()), 
        [data.get(ecoregion, {}).get(land_cover_to_plot, 0) for data in results.values()],
        label=land_cover_to_plot,
        lw=2,  # Line width
        linestyle='-',  # Line style
        marker='o'  # Marker type
    )
    ax.set_title(f'Developed Trends in {ecoregion}')
    ax.set_ylabel('Count')
    ax.legend()

# Disable unused axes if ecoregions do not fill up the entire grid
for i in range(len(ecoregions), len(axes)):
    axes[i].axis('off')

# Set common labels
axes[-1].set_xlabel('Year')  # Set x-label on the last subplot (common x-axis)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract a unique list of all ecoregions from the results
ecoregions = set()
for data in results.values():
    ecoregions.update(data.keys())

# Sort the list for consistent ordering
ecoregions = sorted(ecoregions)

# Define the land cover to plot
land_cover_to_plot = 'Barren'

# Calculate rows needed for the number of ecoregions
num_rows = len(ecoregions) // 4 + (1 if len(ecoregions) % 4 > 0 else 0)

# Setup subplots
fig, axes = plt.subplots(nrows=num_rows, ncols=4, figsize=(20, 3 * num_rows), sharex=True)
axes = axes.flatten()  # Flatten the axes array to simplify indexing

# Loop through each ecoregion and each subplot axis
for ax, ecoregion in zip(axes, ecoregions):
    ax.plot(
        list(total_counts_by_year.keys()), 
        [data.get(ecoregion, {}).get(land_cover_to_plot, 0) for data in results.values()],
        label=land_cover_to_plot,
        lw=2,  # Line width
        linestyle='-',  # Line style
        marker='o'  # Marker type
    )
    ax.set_title(f'Barren Trends in {ecoregion}')
    ax.set_ylabel('Count')
    ax.legend()

# Disable unused axes if ecoregions do not fill up the entire grid
for i in range(len(ecoregions), len(axes)):
    axes[i].axis('off')

# Set common labels
axes[-1].set_xlabel('Year')  # Set x-label on the last subplot (common x-axis)
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm

# ---------------- Paths ----------------
edge_age_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\LCMAP_edge_age_outputs"
edge_west   = os.path.join(edge_age_dir, "EdgeAge_west_2021.tif")
edge_east   = os.path.join(edge_age_dir, "EdgeAge_east_2021.tif")
edge_north  = os.path.join(edge_age_dir, "EdgeAge_north_2021.tif")
edge_south  = os.path.join(edge_age_dir, "EdgeAge_south_2021.tif")

adjunct_lc  = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Edge_adjunct_LC\Adjunct_LC_2021.tif"

# Output CSV
out_csv = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Edge_adjunct_LC\Unexplained_Edge_AdjunctLC_2021_counts.csv"

# ---------------- Land-cover mapping (as integers 0..8) ----------------
lc_map = {
    0: 'Tree Cover',   # (recoded from 4 in your note)
    1: 'Developed',
    2: 'Cropland',
    3: 'Grass/Shrub',
    5: 'Water',
    6: 'Wetland',
    7: 'Ice/Snow',
    8: 'Barren'
}
# Ensure all codes 0..8 exist in the table (missing ones get "Unknown")
all_codes = list(range(0, 9))
name_by_code = {c: lc_map.get(c, f"Unknown_{c}") for c in all_codes}

# ---------------- Parameters ----------------
UNEXPLAINED_VALUE = 34    # pixels with this value in any directional age map are considered "unexplained"
ADJUNCT_MARKER    = 10000 # first digit marks "encoded"

# ---------------- Helper: count blocks for progress bar ----------------
def count_windows(src):
    return sum(1 for _ in src.block_windows(1))

# ---------------- Main ----------------
def main():
    # Open all rasters
    with rasterio.open(adjunct_lc) as adj_src, \
         rasterio.open(edge_west) as w_src, \
         rasterio.open(edge_east) as e_src, \
         rasterio.open(edge_north) as n_src, \
         rasterio.open(edge_south) as s_src:

        # Basic sanity checks
        for ds in (w_src, e_src, n_src, s_src):
            if (ds.width  != adj_src.width or ds.height != adj_src.height or
                ds.transform != adj_src.transform or ds.crs != adj_src.crs):
                raise ValueError("Raster alignment mismatch. Ensure all inputs share the same grid/CRS.")

        # Accumulators for neighbor-class counts (0..8)
        # We'll count *occurrences* among the four directions for masked edge pixels.
        counts = np.zeros(9, dtype=np.int64)

        total_blocks = count_windows(adj_src)
        pbar = tqdm(total=total_blocks, desc="Scanning unexplained edges", unit="block")

        for _, window in adj_src.block_windows(1):
            # Read edge-age windows
            w = w_src.read(1, window=window)
            e = e_src.read(1, window=window)
            n = n_src.read(1, window=window)
            s = s_src.read(1, window=window)

            # Unexplained where any direction has value == 34
            unexplained_mask = (w == UNEXPLAINED_VALUE) | (e == UNEXPLAINED_VALUE) | \
                               (n == UNEXPLAINED_VALUE) | (s == UNEXPLAINED_VALUE)

            # Read adjunct encoding for same window
            enc = adj_src.read(1, window=window)

            # Require valid encoding marker (>=10000)
            valid_enc_mask = enc >= ADJUNCT_MARKER

            # Combine masks: unexplained edges that also have a valid neighbor code
            m = unexplained_mask & valid_enc_mask

            if np.any(m):
                sel = enc[m].astype(np.int64)

                # Strip the marker and decode digits: top, bottom, left, right
                # code format: 10000 + top*1000 + bottom*100 + left*10 + right
                core = sel % 10000
                top    = (core // 1000) % 10
                bottom = (core // 100)  % 10
                left   = (core // 10)   % 10
                right  = (core // 1)    % 10

                # Stack neighbors and count occurrences per class (0..8)
                neigh = np.concatenate([top, bottom, left, right], axis=0)

                # Keep only codes in 0..8
                valid_codes_mask = (neigh >= 0) & (neigh <= 8)
                neigh = neigh[valid_codes_mask]

                if neigh.size > 0:
                    # bincount to vectorize class counts
                    bc = np.bincount(neigh, minlength=9)
                    counts[:9] += bc[:9]

            pbar.update(1)

        pbar.close()

    # Build results table
    total_neigh = counts.sum()
    if total_neigh == 0:
        raise RuntimeError("No adjacent land-cover counts were accumulated. Check inputs and UNEXPLAINED_VALUE.")

    perc = (counts / total_neigh) * 100.0

    df = pd.DataFrame({
        "LC_Code": all_codes,
        "LC_Name": [name_by_code[c] for c in all_codes],
        "Count": counts,
        "Percent": perc
    }).sort_values("Percent", ascending=False)

    # Save & print
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    df.to_csv(out_csv, index=False)
    print("\nAdjunct land cover for unexplained edges (2021)")
    print(df.to_string(index=False, formatters={"Percent": lambda x: f"{x:.2f}"}))
    print(f"\nSaved: {out_csv}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

# your results table pasted here
data = {
    "LC_Code":[0,3,2,6,1,5,8,7,4],
    "LC_Name":["Tree Cover","Grass/Shrub","Cropland","Wetland","Developed","Water","Barren","Ice/Snow","Unknown_4"],
    "Count":[638790752,136027984,128715702,110999998,44024053,10413762,3887392,478665,0],
    "Percent":[59.51,12.67,11.99,10.34,4.10,0.97,0.36,0.04,0.00]
}

df = pd.DataFrame(data)

# filter out Tree Cover (0)
df2 = df[df["LC_Code"] != 0].copy()

# recompute percent without 0
total = df2["Count"].sum()
df2["Percent_Recalc"] = df2["Count"] / total * 100

# pretty print
print(df2[["LC_Code","LC_Name","Count","Percent_Recalc"]]
      .sort_values("Percent_Recalc",ascending=False)
      .to_string(index=False, formatters={"Percent_Recalc":lambda x: f"{x:.2f}"}))
print(f"\nTotal non-forest neighbors counted: {total:,}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Provided data
data = {
    "LC_Code":[0,3,2,6,1,5,8,7,4],
    "LC_Name":["Tree Cover","Grass/Shrub","Cropland","Wetland","Developed","Water","Barren","Ice/Snow","Unknown_4"],
    "Count":[638790752,136027984,128715702,110999998,44024053,10413762,3887392,478665,0]
}

df = pd.DataFrame(data)

# Exclude Tree Cover (0) and Unknown_4 (count=0)
df2 = df[(df["LC_Code"] != 0) & (df["Count"] > 0)].copy()

# Compute recalculated percentage
total = df2["Count"].sum()
df2["Percent"] = df2["Count"] / total * 100

# Sort for prettier display
df2 = df2.sort_values("Percent", ascending=False)

# Color palette
colors = ["#66c2a5","#8da0cb","#a6d854","#ffd92f","#1f78b4","#b3b3b3","#e5c494"][:len(df2)]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    df2["Percent"],
    labels=None,
    autopct='%1.1f%%',
    pctdistance=0.70,   # bring % text inside a bit
    colors=colors
)

# Make % text bigger & readable
for auto in autotexts:
    auto.set_fontsize(11)
    #auto.set_fontweight("bold")

for wedge, label, pct_text in zip(wedges, df2["LC_Name"], autotexts):
    ang = (wedge.theta2 - wedge.theta1) / 2. + wedge.theta1
    x = np.cos(np.deg2rad(ang))
    y = np.sin(np.deg2rad(ang))
    ha = "left" if x > 0 else "right"

    # --- Base positions ---
    x_line = 1.12 * x
    y_line = 1.12 * y
    x_lbl  = 1.24 * x
    y_lbl  = 1.24 * y

    # --- Shift labels + % for tiny slices ---
    if label == "Ice/Snow":   # largest lift
        y_lbl  += 0.15
        y_line += 0.10
        # move % text higher
        x_pct, y_pct = pct_text.get_position()
        pct_text.set_position((x_pct, y_pct + 0.06))

    elif label == "Barren":   # slight lift
        y_lbl  += 0.0
        y_line += 0.0
        x_pct, y_pct = pct_text.get_position()
        pct_text.set_position((x_pct, y_pct ))

    # Draw label and leader line
    ax.text(x_lbl, y_lbl, label, ha=ha, va="center", fontsize=12)
    ax.plot([0.92*x, x_line], [0.92*y, y_line], color='black', linewidth=1)

ax.set_title(
    "Land Cover Adjacent to Unexplained Forest Edges (Tree Cover Removed)",
    fontsize=14
)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----- Input (your tallies) -----
data = {
    "LC_Code":[0,3,2,6,1,5,8,7,4],
    "LC_Name":["Tree Cover","Grass\n&\nShrub","Cropland","Wetland","Developed","Water","Barren","Ice/Snow","Unknown_4"],
    "Count":[638790752,136027984,128715702,110999998,44024053,10413762,3887392,478665,0]
}
df = pd.DataFrame(data)

# ----- Filter: remove Tree Cover & zero-count classes -----
df2 = df[(df["LC_Code"] != 0) & (df["Count"] > 0)].copy()
df2["Percent"] = df2["Count"] / df2["Count"].sum() * 100

# ----- Reorder slices so Barren and Ice/Snow are not adjacent -----
df2 = df2.sort_values("Percent", ascending=False).reset_index(drop=True)
small = df2[df2["LC_Name"].isin(["Barren","Ice/Snow"])]
big   = df2[~df2["LC_Name"].isin(["Barren","Ice/Snow"])]

order_names = []
i_big = i_small = 0
while i_big < len(big) or i_small < len(small):
    if i_big < len(big):
        order_names.append(big.iloc[i_big]["LC_Name"]); i_big += 1
    if i_small < len(small):
        order_names.append(small.iloc[i_small]["LC_Name"]); i_small += 1

df_plot = df2.set_index("LC_Name").loc[order_names].reset_index()

# ----- Plot (donut pie, labels outside, adjusted tiny-slice labels & % texts) -----
fig, ax = plt.subplots(figsize=(8.5, 8.5))
wedges, texts, autotexts = ax.pie(
    df_plot["Percent"],
    labels=None,
    autopct='%1.1f%%',
    pctdistance=0.70,         # pull % inward to avoid overlaps
    startangle=120,           # rotate to open up right-hand space
    wedgeprops={'width':0.45} # donut for cleaner look
)

# Tidy % style
for a in autotexts:
    a.set_fontsize(11)
    #a.set_fontweight("bold")

# Label placement
for wedge, label, pct_text in zip(wedges, df_plot["LC_Name"], autotexts):
    ang = (wedge.theta2 - wedge.theta1) / 2.0 + wedge.theta1
    x = np.cos(np.deg2rad(ang))
    y = np.sin(np.deg2rad(ang))
    ha = "center" if x > 0 else "center"

    # Base anchors
    x_line = 1.10 * x
    y_line = 1.10 * y
    x_lbl  = 1.2 * x
    y_lbl  = 1.2 * y

    # Draw external label and leader line
    ax.text(x_lbl, y_lbl, label, ha=ha, va="center", fontsize=12)
    # two-segment leader (inner to outer arc, then to label)
    ax.plot([0.90*x, x_line], [0.90*y, y_line], color='black', linewidth=1.0)

#ax.set_title("Adjacent  Land Cover to Unexplained Forest Edges", fontsize=14)
plt.tight_layout()
plt.show()